# Gemma 4 26B-A4B Vision Fine-tune on RICO Screen2Words + OASST1

Fine-tunes **`unsloth/gemma-4-26B-A4B-it`** (multimodal MoE) on a 3:1 mix of
[`rootsautomation/RICO-Screen2Words`](https://huggingface.co/datasets/rootsautomation/RICO-Screen2Words)
(image+caption) and [`OpenAssistant/oasst1`](https://huggingface.co/datasets/OpenAssistant/oasst1)
(text-only multi-turn chat). The mix gives screenshot grounding without
losing general conversational ability — needed because the deployed app
analyses both screenshots and free-form messages.

### Key gotchas (verified April 2026)
1. **26B-A4B is MoE → use 16-bit LoRA, NOT 4-bit QLoRA.** 4-bit on MoE silently destroys quality.
2. Needs **>40 GB VRAM** (A100 80 GB / H100). For a smaller GPU drop to `unsloth/gemma-4-E4B-it` — same code, just change the model name and set `load_in_4bit=True, load_in_16bit=False`.
3. Chat template must be `"gemma-4"` at both train and inference time.
4. Vision fine-tuning needs `UnslothVisionDataCollator` and `skip_prepare_dataset=True` — SFT's default text path will drop the PIL images.
5. RICO has 5 captions per screen — we expand each row into 5 training samples (splits dedupe by `screenId` already).
6. OASST samples carry no system prompt and no image, so the model learns to switch between "UI assistant" mode (image + UI system prompt) and "general chat" mode (plain text) based on context.

## 1. Install

In [ ]:
# Uncomment on a fresh machine.
# !pip install --upgrade --no-cache-dir "unsloth>=2026.4" "unsloth_zoo>=2026.4"
# !pip install --no-cache-dir "transformers>=4.50.0" "trl>=0.14.0" "datasets>=3.2.0"
# !pip install --no-cache-dir "peft>=0.14.0" "accelerate>=1.0.0" "bitsandbytes>=0.45.0"
# !pip install --no-cache-dir "evaluate>=0.4.0" "sacrebleu>=2.4.0" "rouge_score>=0.1.2" "nltk>=3.9"
# !pip install --no-cache-dir "clearml>=1.16.0"


## 2. Imports & GPU check

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from clearml import Task
# Re-running this cell in the same kernel? Close any prior task and force a fresh one.
_prev = Task.current_task()
if _prev is not None:
    _prev.close()
clearml_task = Task.init(
    project_name="gemma4",
    task_name="rico_oasst_vision_finetune",
    reuse_last_task_id=False,   # don't reattach to a closed task
    continue_last_task=False,
    output_uri=False,
)
# Defensive: HF's ClearMLCallback rejects 'stopped' tasks. Force in_progress.
clearml_task.mark_started(force=True)

import random
import torch
from PIL import Image
from datasets import load_dataset
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

assert torch.cuda.is_available(), "No GPU detected. Enable a CUDA runtime."
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB")

if vram_gb < 38:
    print("WARNING: <40 GB VRAM. 26B-A4B will likely OOM. Consider E4B.")


## 3. Load Gemma 4 26B-A4B (16-bit LoRA — NOT 4-bit)

In [ ]:
MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/gemma-4-26B-A4B-it"

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,        # MUST be False for 26B-A4B (MoE)
    load_in_16bit=True,        # 16-bit LoRA
    full_finetuning=False,
    # token="hf_...",
)

print(f"[MEM] after load: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

## 4. Attach LoRA (vision + language layers)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=True,        # we want the vision tower to adapt to RICO screenshots
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,                                # MoE is sensitive — start at 16, not 32
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
)
model.print_trainable_parameters()

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

## 5. Load RICO-Screen2Words

In [ ]:
ds = load_dataset("rootsautomation/RICO-Screen2Words")
print(ds)
print("sample columns:", ds["train"].column_names)
print("sample captions:", ds["train"][0]["captions"])

## 6. Build conversation samples

Each row in RICO has 5 reference captions. We expand it into 5 (image, caption)
training pairs. The system prompt frames the model as a mobile-UI assistant
(matches your downstream app), and the user message attaches the screenshot.

In [ ]:
SYSTEM_PROMPT = (
    "You are a mobile UI assistant. You look at app screenshots and describe "
    "what the screen does, what the user can do on it, and answer follow-up "
    "questions. Be specific about UI elements and the screen's purpose."
)

USER_PROMPTS = [
    "Describe what this screen does.",
    "Summarise this app screen in one sentence.",
    "What can a user do on this screen?",
    "What is the purpose of this screen?",
]


def to_conversation(image, caption, prompt=None):
    if prompt is None:
        prompt = random.choice(USER_PROMPTS)
    return {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": caption}]},
        ]
    }


random.seed(3407)

# Optional: subsample train for a faster first run. Set to None for full data.
TRAIN_SUBSAMPLE = None  # e.g. 5000

train_rows = ds["train"]
if TRAIN_SUBSAMPLE:
    train_rows = train_rows.shuffle(seed=3407).select(range(TRAIN_SUBSAMPLE))

rico_convs = []
for row in train_rows:
    img = row["image"].convert("RGB")
    for cap in row["captions"]:
        rico_convs.append(to_conversation(img, cap))

# train_convs is a fresh copy each time this cell runs. Cell 6b appends OASST
# onto rico_convs (not onto train_convs), so re-running 6b is idempotent.
train_convs = list(rico_convs)
print(f"RICO samples: {len(rico_convs)}")
print("Example messages keys:", list(rico_convs[0]["messages"][1]["content"][0].keys()))


## 6b. Mix in OASST1 for general chat ability

RICO captions are short and template-y. Training on RICO alone makes the model
answer follow-up chat questions in the same clipped caption style. We mix in
OpenAssistant/oasst1 multi-turn conversations at a **~3:1 RICO:OASST ratio** —
enough RICO for screenshot grounding, enough OASST to keep the model
conversational for the messaging side of your app.

OASST samples have **no system prompt and no image**, so the model learns to
switch behaviour based on what's in the context (image + UI system prompt → UI
assistant; plain text → general chat).

In [ ]:
# Placeholder image used to keep OASST samples shape-compatible with RICO
# in the same batch. Gemma 4 will resize whatever we give it to ~512x512.
OASST_PLACEHOLDER_IMG = Image.new("RGB", (32, 32), (0, 0, 0))

print("Loading OpenAssistant/oasst1...")
oasst_raw = load_dataset("OpenAssistant/oasst1", split="train")

# Index by message_id and reconstruct conversation chains via parent_id.
msg_by_id = {
    m["message_id"]: {
        "text": m["text"],
        "role": m["role"],          # "prompter" or "assistant"
        "parent_id": m["parent_id"],
        "rank": m["rank"],
    }
    for m in oasst_raw
}


def walk_to_root(message_id):
    chain, current = [], message_id
    while current is not None and current in msg_by_id:
        m = msg_by_id[current]
        chain.append(m)
        current = m["parent_id"]
    chain.reverse()
    return chain


oasst_convs = []
for m in oasst_raw:
    if m["role"] != "assistant":
        continue
    if m["rank"] is not None and m["rank"] > 0:
        continue  # keep best-of-siblings only
    chain = walk_to_root(m["message_id"])
    if len(chain) < 2:
        continue
    # Must alternate prompter, assistant, prompter, ...
    if not all(
        x["role"] == ("prompter" if i % 2 == 0 else "assistant")
        for i, x in enumerate(chain)
    ):
        continue
    msgs = []
    for i, x in enumerate(chain):
        role = "user" if x["role"] == "prompter" else "assistant"
        # Gemma 4 processor requires every sample in a mixed batch to have an
        # image, so attach a tiny black placeholder to the first user turn of
        # each text-only OASST conversation. It costs ~256 vision tokens per
        # sample but lets us mix RICO + OASST in the same batch.
        if i == 0 and role == "user":
            content = [
                {"type": "image", "image": OASST_PLACEHOLDER_IMG},
                {"type": "text", "text": x["text"]},
            ]
        else:
            content = [{"type": "text", "text": x["text"]}]
        msgs.append({"role": role, "content": content})
    oasst_convs.append({"messages": msgs})

print(f"OASST conversations built: {len(oasst_convs)}")

# 3:1 RICO:OASST mix. Subsample OASST to roughly 1/3 of RICO sample count.
target_oasst = max(1, len(train_convs) // 3)
random.shuffle(oasst_convs)
oasst_convs = oasst_convs[:target_oasst]
print(f"OASST kept after subsample: {len(oasst_convs)}")

# Merge and shuffle. IMPORTANT: rebuild from rico_convs (not train_convs) so
# re-running this cell doesn't keep stacking OASST onto an already-mixed list.
mixed_convs = list(rico_convs) + oasst_convs
random.shuffle(mixed_convs)
print(f"Combined training samples: {len(mixed_convs)} "
      f"(RICO={len(rico_convs)}, OASST={len(oasst_convs)})")

# Sanity: every sample must have exactly one image entry (RICO real, OASST placeholder).
def _has_image(conv):
    for msg in conv["messages"]:
        for c in msg["content"]:
            if c.get("type") == "image":
                return True
    return False
_no_img = sum(1 for c in mixed_convs if not _has_image(c))
assert _no_img == 0, f"{_no_img} samples missing images — re-run cell 6 first"
print(f"All {len(mixed_convs)} samples have an image entry. OK to train.")

# Replace train_convs so the training cell below picks up the mixed dataset.
train_convs = mixed_convs


## 7. Train

Uses `UnslothVisionDataCollator` so PIL images survive batching. SFTConfig flags
`skip_prepare_dataset=True` and `remove_unused_columns=False` are mandatory for
vision SFT — without them the trainer will strip the image column.

In [ ]:
OUTPUT_DIR = "gemma4_26b_rico_lora"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_convs,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    args=SFTConfig(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=8,
        warmup_steps=20,
        max_steps=800,                 # bump from 600 since OASST adds ~33% more samples
        learning_rate=1e-4,             # lower than text-only — vision is sensitive
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        report_to="clearml",
        # Vision-required flags:
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ_LENGTH,
    ),
)

stats = trainer.train()
print(f"Final loss: {stats.training_loss:.4f}")

## 8. Save adapter

In [ ]:
ADAPTER_DIR = "gemma4_26b_rico_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to ./{ADAPTER_DIR}")

## 8b. Persist / download the adapter

On Colab everything in `/content/` is wiped when the runtime disconnects.
Three ways to keep the trained adapter (pick one):

1. **Google Drive** — most reliable for multi-GB adapters. Default below.
2. **Hugging Face Hub** — version-controlled and easy to load later.
3. **Direct browser download** — quick zip + `files.download()`. Flaky over ~2 GB.

**What to save:** just the `ADAPTER_DIR` folder. It contains the LoRA weights,
tokenizer, processor, and `adapter_config.json` (which records the base model
name). At inference time, Unsloth re-downloads the base model from HF
automatically — you don't need to ship the 50 GB base.

In [ ]:
import shutil
from pathlib import Path

# --- OPTION 1: Google Drive (recommended on Colab) -----------------------
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DEST = f"/content/drive/MyDrive/{ADAPTER_DIR}"
if os.path.exists(DRIVE_DEST):
    shutil.rmtree(DRIVE_DEST)
shutil.copytree(ADAPTER_DIR, DRIVE_DEST)
print(f"Copied to {DRIVE_DEST}")
print("Size:", sum(f.stat().st_size for f in Path(DRIVE_DEST).rglob('*') if f.is_file()) / 1e9, "GB")


# --- OPTION 2: Hugging Face Hub ------------------------------------------
# from huggingface_hub import login
# login(token="hf_...")  # or set HF_TOKEN env var
# model.push_to_hub(f"your-username/{ADAPTER_DIR}")
# tokenizer.push_to_hub(f"your-username/{ADAPTER_DIR}")


# --- OPTION 3: Direct zip + browser download ----------------------------
# from google.colab import files
# zip_path = shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)
# files.download(zip_path)


# --- OPTIONAL: export merged GGUF for Ollama / llama.cpp / on-device ----
# Big — q4_k_m of a 26B model is ~15 GB. Save straight to Drive.
# model.save_pretrained_gguf(
#     f"/content/drive/MyDrive/{ADAPTER_DIR}_gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
# )


## 9. Evaluation

We evaluate on a held-out slice of `ds["test"]` with two metric families:

1. **Caption quality** — corpus BLEU-4 and ROUGE-L of the model output vs. the 5 reference captions per screen. These are the same metrics the original Screen2Words paper reports.
2. **Follow-up QA sanity** — qualitative pass: feed a screenshot, ask one of the deployment-style questions ("what can the user do here?"), and print the response.

We use a small `EVAL_N` by default because each generation is a forward pass on a 26B model. Bump it for a real benchmark.

In [ ]:
import evaluate

FastModel.for_inference(model)

EVAL_N = 100         # number of test screenshots to score
MAX_NEW_TOKENS = 64  # captions are short
EVAL_PROMPT = "Summarise this app screen in one sentence."

test_rows = ds["test"].shuffle(seed=3407).select(range(EVAL_N))


def generate_caption(image: Image.Image, prompt: str = EVAL_PROMPT) -> str:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()


predictions = []
references = []     # list[list[str]] — 5 refs per row
for i, row in enumerate(test_rows):
    img = row["image"].convert("RGB")
    pred = generate_caption(img)
    predictions.append(pred)
    references.append(list(row["captions"]))
    if i < 5 or i % 20 == 0:
        print(f"[{i:>3}] pred: {pred}")
        print(f"      ref0: {row['captions'][0]}")

In [ ]:
# BLEU-4 (corpus, multi-reference) and ROUGE-L
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

# sacrebleu wants references as a list of lists transposed: refs[ref_idx][example_idx]
# evaluate/sacrebleu wants references shaped [num_examples][num_refs].
max_refs = max(len(r) for r in references)
padded_refs = [list(r) + [""] * (max_refs - len(r)) for r in references]
bleu_score = bleu.compute(predictions=predictions, references=padded_refs)

# ROUGE: compare each prediction against the *best* reference (max-over-refs)
rouge_per_example = []
for pred, refs in zip(predictions, references):
    best = 0.0
    for r in refs:
        s = rouge.compute(predictions=[pred], references=[r])["rougeL"]
        best = max(best, s)
    rouge_per_example.append(best)
rouge_l = sum(rouge_per_example) / len(rouge_per_example)

print("=" * 60)
print(f"BLEU-4 (corpus, 5 refs): {bleu_score['score']:.2f}")
print(f"ROUGE-L (best-of-5):      {rouge_l*100:.2f}")
print("=" * 60)
print("For reference: the original Screen2Words paper reports ~65 BLEU-1 / ~46 BLEU-4")
print("on the full test set with a dedicated screen captioner. This is a quick eval.")


## 9b. Chat eval on OASST validation

Open-ended chat doesn't have a single right answer, so a single metric will
always be noisy. We use two complementary signals on the OASST `validation`
split:

1. **ROUGE-L** between generated reply and the gold final assistant turn —
   correlates with surface-level overlap; trends are meaningful even if
   absolute numbers aren't.
2. **Qualitative samples** — print a few prompts, generations and gold
   responses so you can eyeball whether the model still sounds like a chat
   assistant after the RICO mix-in.

In [ ]:
# Build OASST validation conversation chains the same way as training.
oasst_val = load_dataset("OpenAssistant/oasst1", split="validation")
val_msg_by_id = {
    m["message_id"]: {
        "text": m["text"],
        "role": m["role"],
        "parent_id": m["parent_id"],
        "rank": m["rank"],
    }
    for m in oasst_val
}


def val_walk(mid):
    chain, cur = [], mid
    while cur is not None and cur in val_msg_by_id:
        m = val_msg_by_id[cur]; chain.append(m); cur = m["parent_id"]
    chain.reverse(); return chain


val_chains = []
for m in oasst_val:
    if m["role"] != "assistant":
        continue
    if m["rank"] is not None and m["rank"] > 0:
        continue
    chain = val_walk(m["message_id"])
    if len(chain) < 2:
        continue
    if not all(x["role"] == ("prompter" if i % 2 == 0 else "assistant") for i, x in enumerate(chain)):
        continue
    val_chains.append(chain)
print(f"OASST validation chains: {len(val_chains)}")

CHAT_EVAL_N = 50
random.seed(3407)
random.shuffle(val_chains)
val_chains = val_chains[:CHAT_EVAL_N]


def chat_complete(prefix_msgs):
    """prefix_msgs: list of {role, content:[{type:text,text:...}]} ending with a user turn."""
    inputs = tokenizer.apply_chat_template(
        prefix_msgs,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=200, do_sample=False, use_cache=True,
        )
    plen = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][plen:], skip_special_tokens=True).strip()


chat_preds, chat_golds, chat_prompts = [], [], []
for chain in val_chains:
    # Use everything up to (but not including) the final assistant turn as the prompt.
    gold = chain[-1]["text"]
    prefix = chain[:-1]
    msgs = [
        {
            "role": "user" if x["role"] == "prompter" else "assistant",
            "content": [{"type": "text", "text": x["text"]}],
        }
        for x in prefix
    ]
    pred = chat_complete(msgs)
    chat_preds.append(pred)
    chat_golds.append(gold)
    chat_prompts.append(prefix[-1]["text"])

# ROUGE-L on chat
chat_rouge = rouge.compute(predictions=chat_preds, references=chat_golds)["rougeL"]
print("=" * 60)
print(f"OASST chat ROUGE-L (n={len(chat_preds)}): {chat_rouge*100:.2f}")
print("=" * 60)

# Print a few qualitative samples
print("\nQualitative samples:")
for i in range(min(3, len(chat_preds))):
    print(f"\n--- sample {i} ---")
    print("USER  :", chat_prompts[i][:300])
    print("MODEL :", chat_preds[i][:400])
    print("GOLD  :", chat_golds[i][:400])


## 9c. Base vs finetuned comparison

PEFT lets us toggle the LoRA off in-place via `model.disable_adapter()`,
so we can score the base 26B model against the finetuned one **using the
exact same eval set** without reloading the model. The cell below:

1. Reuses the predictions/refs already computed in cells 9 and 9b for the
   **finetuned** scores.
2. Re-runs the same generations with the LoRA disabled for the **base**
   scores (this doubles eval time — bump `EVAL_N` / `CHAT_EVAL_N` down for a
   quicker turn).
3. Prints a side-by-side delta table and a few qualitative examples.

Note: the base 26B-A4B already captions screens reasonably (it's instruction-
tuned), so don't expect a 0-baseline. The lift is what matters.

In [ ]:
# --- Cache finetuned outputs from cells 9 / 9b -------------------------------
ft_rico_preds   = list(predictions)
ft_chat_preds   = list(chat_preds)
ft_chat_golds   = list(chat_golds)
ft_chat_prompts = list(chat_prompts)
ft_bleu         = bleu_score["score"]
ft_rouge_rico   = rouge_l * 100
ft_chat_rouge   = chat_rouge * 100

# --- Run base (LoRA disabled) on the same eval set ---------------------------
print(f"Generating base predictions on {len(test_rows)} RICO screens...")
base_rico_preds = []
with model.disable_adapter():
    for i, row in enumerate(test_rows):
        img = row["image"].convert("RGB")
        base_rico_preds.append(generate_caption(img))
        if i < 3 or i % 25 == 0:
            print(f"  [{i:>3}] base: {base_rico_preds[-1]}")

    print(f"\nGenerating base predictions on {len(val_chains)} OASST chains...")
    base_chat_preds = []
    for chain in val_chains:
        prefix = chain[:-1]
        msgs = [
            {
                "role": "user" if x["role"] == "prompter" else "assistant",
                "content": [{"type": "text", "text": x["text"]}],
            }
            for x in prefix
        ]
        base_chat_preds.append(chat_complete(msgs))

# --- Score base predictions --------------------------------------------------
base_bleu = bleu.compute(predictions=base_rico_preds, references=padded_refs)["score"]
base_rouge_per = []
for pred, refs in zip(base_rico_preds, references):
    best = 0.0
    for r in refs:
        best = max(best, rouge.compute(predictions=[pred], references=[r])["rougeL"])
    base_rouge_per.append(best)
base_rouge_rico = sum(base_rouge_per) / len(base_rouge_per) * 100
base_chat_rouge = rouge.compute(predictions=base_chat_preds, references=ft_chat_golds)["rougeL"] * 100

# --- Comparison table --------------------------------------------------------
print("\n" + "=" * 60)
print(f"{'metric':<22}{'base':>10}{'finetuned':>14}{'delta':>10}")
print("-" * 60)
def _row(label, b, f):
    print(f"{label:<22}{b:>10.2f}{f:>14.2f}{f - b:>+10.2f}")
_row("RICO BLEU-4",          base_bleu,        ft_bleu)
_row("RICO ROUGE-L",         base_rouge_rico,  ft_rouge_rico)
_row("OASST chat ROUGE-L",   base_chat_rouge,  ft_chat_rouge)
print("=" * 60)

# --- Side-by-side qualitative samples ---------------------------------------
print("\nRICO samples (REF / BASE / FT):")
for i in range(min(3, len(base_rico_preds))):
    print(f"\n--- rico {i} ---")
    print("REF :", test_rows[i]["captions"][0])
    print("BASE:", base_rico_preds[i])
    print("FT  :", ft_rico_preds[i])

print("\nOASST chat samples (USER / BASE / FT / GOLD):")
for i in range(min(3, len(base_chat_preds))):
    print(f"\n--- oasst {i} ---")
    print("USER:", ft_chat_prompts[i][:250])
    print("BASE:", base_chat_preds[i][:350])
    print("FT  :", ft_chat_preds[i][:350])
    print("GOLD:", ft_chat_golds[i][:350])


## 10. Qualitative test — your mobile-app use case

Two checks that match how the deployed app will call the model:

- **Free-form description** — same as eval prompt.
- **Follow-up question** — multi-turn, image stays in context.

In [ ]:
def chat_about_screenshot(image: Image.Image, turns: list[str]) -> list[str]:
    """Run a multi-turn conversation about a single screenshot."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": turns[0]},
        ]},
    ]
    replies = []
    for i, user_text in enumerate(turns):
        if i > 0:
            messages.append({"role": "user", "content": [{"type": "text", "text": user_text}]})
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        ).to("cuda")
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=160, do_sample=False, use_cache=True)
        reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        replies.append(reply)
        messages.append({"role": "assistant", "content": [{"type": "text", "text": reply}]})
    return replies


sample = ds["test"][0]
img = sample["image"].convert("RGB")
turns = [
    "Describe what this screen does.",
    "What is the most likely action a user takes here?",
    "List the visible UI controls.",
]
replies = chat_about_screenshot(img, turns)
for q, a in zip(turns, replies):
    print(f"USER: {q}")
    print(f"BOT : {a}\n")
print("GROUND-TRUTH CAPTIONS:")
for c in sample["captions"]:
    print("  -", c)

## 11. (optional) Test on your own app screenshots

Drop a PNG/JPG path here to sanity-check on screens that aren't in RICO.

In [ ]:
# CUSTOM_IMAGE = "/path/to/your_app_screenshot.png"
# img = Image.open(CUSTOM_IMAGE).convert("RGB")
# print(generate_caption(img, "Describe what this screen does."))